In [ ]:
#| default_exp write

# write

> Appending to a ledger. One call, one line, no lock.

`Scribe` is the whole writing surface. Every method appends one record and returns; nothing is
held open between calls and nothing is rewritten. That is what lets a Claude Code hook, a Codex
hook and a Ramabana turn write to the same ledger from three processes without arranging
anything between them.

A session is assembled from records rather than stored as one. `begin` says a session started,
`step` and `touch` say what it did, `end` says how it finished. A reader folds them.

In [ ]:
#| export
import re
from pathlib import Path

from fastcore.basics import AttrDict, ifnone

from fastcore.basics import patch

from gheasy.repo import GitRepo

from panjika.core import (DETAIL, LEDGER, MAX_DETAIL, Home, append, file_hash, git_root, hashed,
                   new_id, now)

## What a tool did

A step names an action as well as a tool, because the tool names differ between harnesses and
the actions do not. `Edit`, `apply_patch` and `edit_file` are three names for `write`.

In [ ]:
#| export
#: action -> the substrings of a tool name that mean it. First match wins, so order matters.
ACTION_HINTS = (
    ('write', ('edit', 'write', 'patch', 'create', 'apply', 'update', 'insert', 'replace', 'delete')),
    ('run',   ('bash', 'shell', 'exec', 'run', 'terminal', 'command', 'python', 'notebook_run')),
    ('net',   ('web', 'fetch', 'url', 'browser', 'http', 'research', 'crawl')),
    ('search',('grep', 'search', 'glob', 'find', 'index', 'similar', 'outline', 'symbols')),
    ('read',  ('read', 'view', 'cat', 'open', 'ls', 'list', 'show', 'inspect', 'vars')),
    ('plan',  ('todo', 'plan', 'task')),
    ('ask',   ('ask', 'approval', 'question', 'permission')),
)

ACTIONS = ('write', 'run', 'net', 'search', 'read', 'plan', 'ask', 'other')


def action_for(tool):
    "Which of `ACTIONS` a tool name means. `other` when nothing matches."
    name = str(tool or '').lower()
    for action, hints in ACTION_HINTS:
        if any(h in name for h in hints): return action
    return 'other'

In [ ]:
from fastcore.test import test_eq
test_eq(action_for('Edit'), 'write')
test_eq(action_for('apply_patch'), 'write')
test_eq(action_for('Bash'), 'run')
test_eq(action_for('WebFetch'), 'net')
test_eq(action_for('Grep'), 'search')
test_eq(action_for('Read'), 'read')
test_eq(action_for('mcp__something__weird'), 'other')

## The scribe

In [ ]:
#| export
def _target(value, n=160):
    "The one-line target of a step: a path, a command, a url, whatever it was pointed at."
    return ' '.join(str(value or '').split())[:n]


class Scribe:
    """Appends to one ledger.

    `home` is found from `start` when it is not given, so a hook in a repository writes to that
    repository's ledger without being told where it is.
    """

    def __init__(self,
                 home=None,         # the ledger folder. None finds one from `start`
                 session='',        # the session every record joins. None starts a fresh id
                 start='.',         # where to look for a ledger, and the repository to describe
                 detail=True):      # write the machine-local tier as well as the committed one
        self.home = home if isinstance(home, Home) else Home(home, start)
        self.session = str(session or new_id('s-'))
        self.start, self.detail = str(start), bool(detail)
        self._seq = 0

    def __repr__(self): return f'Scribe({self.session} -> {self.home.path})'

    def write(self, kind, detail=None, **fields):
        """Append one record. `detail` goes to the machine-local tier under the same id.

        Every record gets an `id`, which is only ever used to drop a line that reached the
        ledger twice. A caller that can name an event stably -- a backfill re-reading the same
        transcript -- passes its own, and re-recording that event changes nothing. Records
        belong to a session through `session`, so a session can be described by as many
        records as it takes.
        """
        rec = {'kind': kind, 'id': fields.pop('id', None) or new_id(),
               'at': fields.pop('at', None) or now(),
               'session': fields.pop('session', None) or self.session, **fields}
        rec = {k: v for k, v in rec.items() if v is not None}
        out = append(self.home.shard(LEDGER, rec['at']), rec)
        if detail and self.detail:
            append(self.home.shard(DETAIL, rec['at']),
                   {'kind': kind, 'id': rec['id'], 'at': rec['at'], 'session': rec['session'], **detail},
                   MAX_DETAIL)
        return out

### Beginning and ending

`begin` describes the session: which harness, which model, what it was asked. It fills in the
repository, the branch and the working directory itself, because a hook rarely knows them and a
trail is not much use without them.

In [ ]:
#| export
def repo_facts(start='.'):
    "The repository name, root and current branch for `start`, as far as they can be told."
    root = git_root(start)
    branch = ''
    if root is not None:
        # gheasy has no public accessor for the checked-out branch, so ask git through its own
        try: branch = GitRepo(root).run('rev-parse', '--abbrev-ref', 'HEAD').strip()
        except Exception: branch = ''
    return {'repo': root.name if root else '', 'root': str(root) if root else '',
            'branch': '' if branch == 'HEAD' else branch}

In [ ]:
#| export
@patch
def begin(self:Scribe, harness, model='', prompt='', title='', parent='', agent='', **fields):
    "Record that a session started. Returns its id."
    self.write('session', harness=str(harness), model=str(model or ''),
               prompt=_target(prompt, 2000), title=str(title or ''), status='open',
               parent=str(parent or ''), agent=str(agent or ''),
               started=fields.pop('started', None) or fields.get('at') or now(),
               cwd=str(Path(self.start).resolve()), **repo_facts(self.start), **fields)
    return self.session


@patch
def end(self:Scribe, status='done', **fields):
    "Record that a session finished, with whatever counters the harness can supply."
    return self.write('session', status=str(status),
                      ended=fields.pop('ended', None) or fields.get('at') or now(), **fields)

### Steps

The ledger tier keeps the fact of the call: which tool, what it pointed at, whether it worked,
how long it took. Whole arguments and whole output go to the detail tier under the same id, so
the committed half stays a summary and nothing is lost locally.

In [ ]:
#| export
@patch
def step(self:Scribe, tool, target='', ok=True, secs=0.0, action='', summary='',
         args=None, output=None, **fields):
    "Record one tool call. Returns the step id."
    self._seq += 1
    detail = None
    if args is not None or output is not None:
        detail = {'tool': str(tool), 'args': args, 'output': None if output is None else str(output)}
    rec = self.write('step', detail=detail, tool=str(tool), seq=self._seq,
                     action=action or action_for(tool), target=_target(target),
                     ok=bool(ok), secs=round(float(secs or 0), 3),
                     summary=_target(summary, 300), **fields)
    return rec.id


@patch
def note(self:Scribe, text, **fields):
    "Record a line of prose against the session: a plan, a summary, a reason."
    return self.write('note', text=str(text)[:4000], **fields).id

### Touches

A touch is the interesting record, because it is the one the landing check reads back. It holds
the file, what was done to it, the hash of the file at `HEAD`, the hash of it now, and the shape
of the difference between them.

The hashes of the lines the change added go to the detail tier. They are what makes the landing
check exact later: a line can be found again after it has moved, and `git blame` will say which
commit owns it. The ledger tier keeps only the counts, so the committed half stays small and
carries no source.

In [ ]:
#| export
def text_changes(before, after):
    "The lines `after` adds to `before` and the ones it drops, for a change git cannot be asked about."
    import difflib
    b, a = (before or '').splitlines(), (after or '').splitlines()
    add, rem = [], []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, b, a).get_opcodes():
        if tag in ('replace', 'delete'): rem += b[i1:i2]
        if tag in ('replace', 'insert'): add += a[j1:j2]
    return add, rem


def diff_lines(diff):
    """The added and removed lines of a unified diff, without its headers.

    A header is `+++ ` or `--- ` with its space. Testing for `+++` alone drops a body line
    whose own text starts with `++`, and the same edit then counts differently depending on
    whether a hook or a backfill recorded it.
    """
    added, removed = [], []
    for line in str(diff or '').splitlines():
        if line[:4] in ('+++ ', '--- ') or line in ('+++', '---'): continue
        if line.startswith('+'): added.append(line[1:])
        elif line.startswith('-'): removed.append(line[1:])
    return added, removed


def head_hash(root, path):
    "The hash of a path's committed content at HEAD, or `''` when it is not committed."
    try: return hashed(GitRepo(root).run('show', f'HEAD:{path}'))
    except Exception: return ''

In [ ]:
from fastcore.test import test_eq

# A hook reads a unified diff and a backfill reads the two bodies. They have to hash the same
# strings or a backfilled session stops matching `git blame` and every verdict for it quietly
# degrades -- nothing raises, the answers just get worse. A header is `+++ ` with its space, so
# testing for `+++` alone would eat a body line whose own text starts with `++`.
before = 'int x;\n'
after = 'int x;\n++counter;\n\tindented = 3\ntrailing = 4   \n'
diff = ('--- a/f.c\n+++ b/f.c\n@@ -1,1 +1,4 @@\n int x;\n'
        '+++counter;\n+\tindented = 3\n+trailing = 4   \n')
test_eq(diff_lines(diff)[0], text_changes(before, after)[0])
test_eq(diff_lines(diff)[0], ['++counter;', '\tindented = 3', 'trailing = 4   '])

In [ ]:
#| export
@patch
def touch(self:Scribe, path, action='edit', step='', before=None, after_text=None, **fields):
    """Record that a file was touched, and how it now differs from `HEAD`.

    The difference is measured against `HEAD` rather than against the previous touch, so a
    session that edits one file five times leaves five records of the same net change and a
    reader can keep the last without arithmetic.

    `before` and `after_text` are for a backfill, where the working tree has moved on and git
    can no longer say what the session did. Given them, the change is read from the bodies the
    transcript carries instead of from the repository.
    """
    p = Path(path)
    root = git_root(p if p.is_absolute() else self.start)
    rel = str(p.resolve().relative_to(root)) if root and p.is_absolute() else str(path)
    ranges, added, removed, tracked = [], [], [], bool(root)
    if after_text is not None:
        added, removed = text_changes(before, after_text)
    elif root is not None:
        try:
            ch = GitRepo(root).file_changes(rel)
            ranges = [[r['from'], r['to'], r['kind']] for r in ch.get('ranges') or []]
            added, removed = diff_lines(ch.get('diff'))
            tracked = not (ch.get('status') or {}).get('untracked', False)
        except Exception: pass
    detail = {'path': rel, 'lines': [hashed(t) for t in added][:2000]} if added else None
    rec = self.write('touch', detail=detail, path=rel, action=str(action), step=str(step or ''),
                     head=head_hash(root, rel) if root else '',
                     after=(hashed(after_text) if after_text is not None else
                            file_hash(p if p.is_absolute() else Path(self.start)/path)),
                     added=len(added), removed=len(removed), ranges=ranges[:200],
                     tracked=tracked, **fields)
    return rec.id


@patch
def commit(self:Scribe, sha, subject='', author='', files=(), branch='', **fields):
    "Record that a commit exists and which session it belongs to."
    return self.write('commit', sha=str(sha), subject=_target(subject, 300), author=str(author or ''),
                      files=list(files)[:500], branch=str(branch or ''), **fields).id

## A whole session, written

In [ ]:
import subprocess, tempfile
from panjika.core import Home, records, fold

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

sc = Scribe(home=d/'.panjika', start=d)
sc.home.init()
sc.begin('claude-code', model='opus-5', prompt='make add handle strings')
(d/'app.py').write_text('def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
sid = sc.session
step = sc.step('Edit', target='app.py', secs=0.4, args={'file_path': 'app.py'})
sc.touch(d/'app.py', 'edit', step)
sc.end('done', turns=1, steps_ok=1, steps_fail=0)

rows = {r.kind: r for r in records(sc.home)}
test_eq(sorted(rows), ['session', 'step', 'touch'])   # two session records, begin and end
test_eq(rows['touch'].path, 'app.py')
test_eq(rows['touch'].added, 1)
test_eq(fold(records(sc.home).filter(lambda r: r.kind == 'session'))[0].status, 'done')

In [ ]:
# the committed tier carries no source, and the machine-local tier carries the line hashes
from panjika.core import DETAIL
ledger_text = sc.home.shard().read_text()
assert 'isinstance' not in ledger_text, 'the ledger tier must not carry source'
detail = records(sc.home, DETAIL).filter(lambda r: r.kind == 'touch')
test_eq(len(detail[0].lines), 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()